IMPORT LIBRARY

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm

In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

True

LOAD DATASET

In [ ]:
df = pd.read_csv('https://drive.google.com/uc?export=download&id=1lWGOzr20bgHEf81abiICvcXGyyO3Hos4')
df.head()

,review,sentiment
0,This is an utterly forgettable picture. A frie...,negative
1,"Very suspenseful, surprisingly intelligent fil...",positive
2,"""Journey to the Far Side of the Sun"" (aka ""Dop...",positive
3,During the whole Pirates of The Caribbean Tril...,positive
4,I can't come up with appropriate enough words ...,negative


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     20000 non-null  object
 1   sentiment  20000 non-null  object
dtypes: object(2)
memory usage: 312.6+ KB


CASE FOLDING

In [ ]:
df['clean_review'] = df['review'].str.lower()
df[['review', 'clean_review']].head(5)

,review,clean_review
0,This is an utterly forgettable picture. A frie...,this is an utterly forgettable picture. a frie...
1,"Very suspenseful, surprisingly intelligent fil...","very suspenseful, surprisingly intelligent fil..."
2,"""Journey to the Far Side of the Sun"" (aka ""Dop...","""journey to the far side of the sun"" (aka ""dop..."
3,During the whole Pirates of The Caribbean Tril...,during the whole pirates of the caribbean tril...
4,I can't come up with appropriate enough words ...,i can't come up with appropriate enough words ...


DATA CLEANING

In [ ]:
def clean_data(text):
    text = text.str.strip()
    text = text.str.replace(r'[^a-zA-Z\s]', '', regex=True)
    text = text.str.replace(r' {2,}', ' ', regex=True).str.strip()
    text = text.str.replace(r'<br />', '', regex=True)
    text = text.str.replace(r'http\S+', '', regex=True)

    return text

df['clean_review'] = clean_data(df['clean_review'])
df[['review', 'clean_review']].head(5)

,review,clean_review
0,This is an utterly forgettable picture. A frie...,this is an utterly forgettable picture a frien...
1,"Very suspenseful, surprisingly intelligent fil...",very suspenseful surprisingly intelligent film...
2,"""Journey to the Far Side of the Sun"" (aka ""Dop...",journey to the far side of the sun aka doppelg...
3,During the whole Pirates of The Caribbean Tril...,during the whole pirates of the caribbean tril...
4,I can't come up with appropriate enough words ...,i cant come up with appropriate enough words t...


TOKENIZATION

In [ ]:
df['tokens'] = df['clean_review'].str.split()
df[['clean_review', 'tokens']].head(5)

,clean_review,tokens
0,this is an utterly forgettable picture a frien...,"[this, is, an, utterly, forgettable, picture, ..."
1,very suspenseful surprisingly intelligent film...,"[very, suspenseful, surprisingly, intelligent,..."
2,journey to the far side of the sun aka doppelg...,"[journey, to, the, far, side, of, the, sun, ak..."
3,during the whole pirates of the caribbean tril...,"[during, the, whole, pirates, of, the, caribbe..."
4,i cant come up with appropriate enough words t...,"[i, cant, come, up, with, appropriate, enough,..."


STOPWORD REMOVAL

In [ ]:
stopwords_english = set(stopwords.words('english'))
custom_stopwords = stopwords_english.union({'movie', 'film', 'one', 'make', 'even', 'see', 'watch', 'time', 'character', 'story'})

In [ ]:
def remove_stopwords(tokens_list):
    return [word for word in tokens_list if word not in custom_stopwords]

In [ ]:
tqdm.pandas(desc="Menghapus Stopword")
df['tokens_no_stopword'] = df['tokens'].progress_apply(remove_stopwords)
df[['tokens', 'tokens_no_stopword']].head(5)

Menghapus Stopword: 100%|██████████| 20000/20000 [00:00<00:00, 39884.60it/s]


,tokens,tokens_no_stopword
0,"[this, is, an, utterly, forgettable, picture, ...","[utterly, forgettable, picture, friend, mine, ..."
1,"[very, suspenseful, surprisingly, intelligent,...","[suspenseful, surprisingly, intelligent, five,..."
2,"[journey, to, the, far, side, of, the, sun, ak...","[journey, far, side, sun, aka, doppelganger, e..."
3,"[during, the, whole, pirates, of, the, caribbe...","[whole, pirates, caribbean, trilogy, craze, pa..."
4,"[i, cant, come, up, with, appropriate, enough,...","[cant, come, appropriate, enough, words, descr..."


LEMMATIZATION

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
def apply_lemmatization(tokens_list):
    return [lemmatizer.lemmatize(word) for word in tokens_list]

In [ ]:
tqdm.pandas(desc="Lemmatization")
df['lemmatized_tokens'] = df['tokens_no_stopword'].progress_apply(apply_lemmatization)
df[['tokens_no_stopword', 'lemmatized_tokens']].head(5)

Lemmatization: 100%|██████████| 20000/20000 [00:12<00:00, 1550.49it/s]


,tokens_no_stopword,lemmatized_tokens
0,"[utterly, forgettable, picture, friend, mine, ...","[utterly, forgettable, picture, friend, mine, ..."
1,"[suspenseful, surprisingly, intelligent, five,...","[suspenseful, surprisingly, intelligent, five,..."
2,"[journey, far, side, sun, aka, doppelganger, e...","[journey, far, side, sun, aka, doppelganger, e..."
3,"[whole, pirates, caribbean, trilogy, craze, pa...","[whole, pirate, caribbean, trilogy, craze, par..."
4,"[cant, come, appropriate, enough, words, descr...","[cant, come, appropriate, enough, word, descri..."


TOKEN N LEMMA

In [ ]:
import spacy

In [ ]:
nlp = spacy.load('en_core_web_sm')
phrase = df.loc[1, 'clean_review']
phrase

'very suspenseful surprisingly intelligent film about five medical students flatlining themselves and then being resuscitated to share their experiences of death and what lies beyond joel schumacher directs with some skill creating some very eerie scenes as well as particularly beautiful ones the visions of death are not what viewers might expect nor is that which awaits us all when we go thanks to screenwriter peter filardi who really did an outstanding job coming up with this story while the creativity of the story is impressive the story has many holes as well particularly in the logic department and believability factors notwithstanding all of that flatliners is a good effective film because of the script the direction which again is very surreal at times and the acting which brings four very talented actors and william baldwin together this core of actors acts and reacts off each other very nicely keifer sutherland does a very impressive job as the head of the group the one who co

In [ ]:
doc = nlp(phrase)
doc

very suspenseful surprisingly intelligent film about five medical students flatlining themselves and then being resuscitated to share their experiences of death and what lies beyond joel schumacher directs with some skill creating some very eerie scenes as well as particularly beautiful ones the visions of death are not what viewers might expect nor is that which awaits us all when we go thanks to screenwriter peter filardi who really did an outstanding job coming up with this story while the creativity of the story is impressive the story has many holes as well particularly in the logic department and believability factors notwithstanding all of that flatliners is a good effective film because of the script the direction which again is very surreal at times and the acting which brings four very talented actors and william baldwin together this core of actors acts and reacts off each other very nicely keifer sutherland does a very impressive job as the head of the group the one who com

In [ ]:
[token for token in doc]

[very,
 suspenseful,
 surprisingly,
 intelligent,
 film,
 about,
 five,
 medical,
 students,
 flatlining,
 themselves,
 and,
 then,
 being,
 resuscitated,
 to,
 share,
 their,
 experiences,
 of,
 death,
 and,
 what,
 lies,
 beyond,
 joel,
 schumacher,
 directs,
 with,
 some,
 skill,
 creating,
 some,
 very,
 eerie,
 scenes,
 as,
 well,
 as,
 particularly,
 beautiful,
 ones,
 the,
 visions,
 of,
 death,
 are,
 not,
 what,
 viewers,
 might,
 expect,
 nor,
 is,
 that,
 which,
 awaits,
 us,
 all,
 when,
 we,
 go,
 thanks,
 to,
 screenwriter,
 peter,
 filardi,
 who,
 really,
 did,
 an,
 outstanding,
 job,
 coming,
 up,
 with,
 this,
 story,
 while,
 the,
 creativity,
 of,
 the,
 story,
 is,
 impressive,
 the,
 story,
 has,
 many,
 holes,
 as,
 well,
 particularly,
 in,
 the,
 logic,
 department,
 and,
 believability,
 factors,
 notwithstanding,
 all,
 of,
 that,
 flatliners,
 is,
 a,
 good,
 effective,
 film,
 because,
 of,
 the,
 script,
 the,
 direction,
 which,
 again,
 is,
 very,
 surre

In [ ]:
[token.lemma_ for token in doc]

['very',
 'suspenseful',
 'surprisingly',
 'intelligent',
 'film',
 'about',
 'five',
 'medical',
 'student',
 'flatline',
 'themselves',
 'and',
 'then',
 'be',
 'resuscitate',
 'to',
 'share',
 'their',
 'experience',
 'of',
 'death',
 'and',
 'what',
 'lie',
 'beyond',
 'joel',
 'schumacher',
 'direct',
 'with',
 'some',
 'skill',
 'create',
 'some',
 'very',
 'eerie',
 'scene',
 'as',
 'well',
 'as',
 'particularly',
 'beautiful',
 'one',
 'the',
 'vision',
 'of',
 'death',
 'be',
 'not',
 'what',
 'viewer',
 'might',
 'expect',
 'nor',
 'be',
 'that',
 'which',
 'await',
 'we',
 'all',
 'when',
 'we',
 'go',
 'thank',
 'to',
 'screenwriter',
 'peter',
 'filardi',
 'who',
 'really',
 'do',
 'an',
 'outstanding',
 'job',
 'come',
 'up',
 'with',
 'this',
 'story',
 'while',
 'the',
 'creativity',
 'of',
 'the',
 'story',
 'be',
 'impressive',
 'the',
 'story',
 'have',
 'many',
 'hole',
 'as',
 'well',
 'particularly',
 'in',
 'the',
 'logic',
 'department',
 'and',
 'believability'

In [ ]:
df.columns

Index(['review', 'sentiment', 'clean_review', 'tokens', 'tokens_no_stopword',
       'lemmatized_tokens'],
      dtype='object')

In [ ]:
df['clean_text'] = df['lemmatized_tokens'].apply(lambda x: " ".join(x))

## Feature Extraction

tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

y = df['sentiment']

In [ ]:
print("Shape TF-IDF:", X_tfidf.shape)

Shape TF-IDF: (20000, 5000)


In [ ]:
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
tfidf_df.head()

,abandoned,abc,ability,able,abortion,aboutbr,absence,absent,absolute,absolutely,...,youll,young,younger,youngster,youre,youth,youve,zero,zombie,zone
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.062934,0.0,0.077325,0.000000,0.0,0.000000
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.062115,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.085626
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000
4,0.0,0.0,0.0,0.049263,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.045074,0.0,0.000000,0.065362,0.0,0.000000


word embedding

In [ ]:
import sys
!{sys.executable} -m pip install gensim
from gensim.models import Word2Vec

# ambil token
sentences = df['lemmatized_tokens']

# training model
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [ ]:
import numpy as np

def sentence_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

X_w2v = np.array([sentence_vector(tokens, w2v_model) for tokens in sentences])

print("Shape Word2Vec:", X_w2v.shape)

Shape Word2Vec: (20000, 100)


In [ ]:
print("Contoh vector Word2Vec:")
print(X_w2v[0])

Contoh vector Word2Vec:
[-0.23617195  0.09999529  0.05723929 -0.00556858 -0.01586677 -0.58259034
  0.3118515   1.1050868  -0.6580799  -0.28573057 -0.16166672 -0.6793917
  0.28287598  0.12532139  0.15887712  0.09260058  0.25247222 -0.43050355
 -0.08071463 -1.115229    0.30789822  0.10586035  0.40268105 -0.54934627
 -0.1520186   0.09335429 -0.35236233 -0.34096295 -0.00184129  0.01631567
  0.612201    0.02852806  0.38227424 -0.3185724  -0.23924465  0.8034991
  0.26909277 -0.08811619 -0.16038188 -0.6660128   0.27322808 -0.56431204
 -0.21393359  0.1186506   0.08240961 -0.14035852 -0.4149326   0.05441326
  0.3236705   0.09231081 -0.00830953 -0.19409785 -0.04580628  0.11495141
 -0.08073205  0.1180449   0.4319619  -0.02450562 -0.22591281  0.41856232
  0.04895643  0.14177078  0.21737562 -0.09905497 -0.48089772  0.43222246
  0.14761673  0.3090695  -0.36880934  0.40446118 -0.269783    0.07545861
  0.42761087  0.30037108  0.6589445   0.03131417  0.11364658  0.12228939
 -0.10644054 -0.01702153 -0.2

In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
negative,10000
positive,10000
